# Large-Scale Thematic Research Workflow (Client Example)

Client-ready workflow on [Bigdata.com](https://bigdata.com) to build auditable thematic signals from unstructured content across a broad equity universe.

This notebook supports two signal families with an explicit accuracy and recall trade-off:

1. **Basic signal (high recall, lower precision, fast)**
2. **Advanced signal (higher precision, slower, with LLM validation)**



**Create a `.env` file in this folder before running the notebook**

Required:

`BIGDATA_API_KEY=your_api_key`

Optional:

- `BIGDATA_API_BASE_URL=...` (override API endpoint)
- `OPENAI_API_KEY=...` (needed for LLM validation stage)

Notes:
- Restart the kernel after changing credentials.
- Keep credentials out of source control.



In [124]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Imports



In [1]:
import asyncio
import json
import logging
import os
import pickle
import sys
from pathlib import Path
from zoneinfo import ZoneInfo

import nest_asyncio
import pandas as pd
import plotly.io as pio
from dotenv import load_dotenv

from src import (
    BigDataSession,
    convert_to_dataframe,
    execute_full_grid_search,
    aggregate_results_by_chunk,
    extract_companies_from_entity_list,
    get_unknown_entities_from_df_column,
    keep_only_companies_in_detections,
    map_create_only_companies_column,
    explode_to_dataframe,
)

from bigdata_smart_batching import (plan_search, 
save_plan, execute_search, 
deduplicate_documents, load_plan

)
import importlib
import src.helper as helper_mod

importlib.reload(helper_mod)
build_rolling_impact_signal = helper_mod.build_rolling_impact_signal
plot_top_entities_rolling_signal = helper_mod.plot_top_entities_rolling_signal
mask_companies_in_df = helper_mod.mask_companies_in_df

from src.labeler.screener_labeler import Labeler, merge_validation_labels

from src.mindmap import generate_risk_tree, generate_theme_tree


## Configuration



In [2]:
# Suppress verbose HTTP logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

# Load environment variables
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)

# Add current directory to path for local imports
sys.path.insert(0, str(Path.cwd()))

# Apply nest_asyncio if needed (for Jupyter environments)
try:
    asyncio.get_running_loop()
    nest_asyncio.apply()
    print("✅ nest_asyncio applied")
except RuntimeError:
    pass  # No running loop, nest_asyncio not needed

# Configure Plotly renderer
try:
    if 'JUPYTERHUB_SERVICE_PREFIX' in os.environ or 'JPY_SESSION_NAME' in os.environ:
        pio.renderers.default = 'jupyterlab'
    else:
        pio.renderers.default = 'plotly_mimetype+notebook'
except Exception:
    pio.renderers.default = 'notebook'



✅ nest_asyncio applied


## Setup & Authentication

This section loads environment credentials from `.env` and initializes `session`.

Expected outcomes:
- authenticated Bigdata session,
- stable API base URL,
- optional OpenAI key availability for advanced (LLM-verified) signal path.



In [3]:
# Setup & Authentication (single source of truth)
# Loads credentials from .env in the notebook folder.
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)

BIGDATA_API_KEY = os.getenv("BIGDATA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

try:
    session = BigDataSession()
    print("✅ API key configured")
    print(f"✅ API Base URL: {session.api_base_url}")
except ValueError as e:
    print(f"❌ Authentication failed: {e}")
    print("   Please set BIGDATA_API_KEY in your .env file")
    session = None


✅ API key configured
✅ API Base URL: https://api.bigdata.com


## Search Parameters

Define the core experiment scope:
- thematic hypothesis,
- investable universe,
- analysis window,
- retrieval sampling controls.

These settings directly affect recall, compute cost, and downstream signal stability.



In [4]:
# Main theme to analyze - will be decomposed into sub-themes for higher recall
theme = "Semiconductor supply shortage disrupted global production"

# Universe: CSV file with columns 'id' (entity ID) and 'name' (company name)
universe_csv = "id_name_mapping_us_top_3000.csv"

# Time window for the search
start_date = "2021-01-01"
end_date = "2021-06-30"

# Sampling percentage: controls the % of chunks to retrieve per basket (0.0-1.0)
# Lower values = faster execution, higher values = more comprehensive results
chunk_percentage = 0.2


## 1. Theme Decomposition

A single query under-covers real-world language variation. We decompose the main theme into terminal sub-themes to improve retrieval coverage.

Why this matters:
- higher recall across different phrasing of the same economic mechanism,
- better separation of causal channels (e.g., delays, margin pressure, cancellations).

Output artifact:
- `terminal_labels_and_summaries` used as independent search queries.



In [7]:
# Generate risk tree from main theme using LLM
# Alternative: use generate_theme_tree(main_theme=theme) for general thematic decomposition
risk_tree = generate_risk_tree(
    main_theme=theme
)

# Extract terminal nodes: these are the specific sub-themes we'll search for
terminal_labels_and_summaries = risk_tree.get_terminal_label_summaries()



In [8]:
terminal_labels_and_summaries


{'Production Delays': 'Companies will face production delays due to semiconductor supply shortages.',
 'Increased Lead Times': 'Firms will experience increased lead times for products reliant on semiconductors.',
 'Supplier Instability': 'Supplier instability will arise as firms struggle to secure semiconductor components.',
 'Rising Input Costs': 'Companies will face rising input costs due to semiconductor shortages.',
 'Increased Operational Costs': 'Operational costs will increase as firms seek alternative semiconductor sources.',
 'Price Inflation': 'Price inflation will occur as demand for semiconductors outstrips supply.',
 'Loss of Market Share': 'Firms may lose market share due to inability to produce goods without semiconductors.',
 'Reduced Product Offerings': 'Companies will reduce product offerings as semiconductor shortages limit production.',
 'Geographic Market Limitations': 'Geographic market limitations will arise as firms cannot meet local demand for products.',
 'Cus

In [5]:
# For faster testing, only use the first item from terminal_labels_and_summaries
terminal_labels_and_summaries = {'Production Delays':'Companies will face production delays due to semiconductor supply shortages disrupting global production.'}

## 2. Search Planning with Smart Batching

`plan_search(...)` builds a query plan that allocates retrieval budget across entities/time based on expected content density.

Why this matters:
- avoids over-allocating budget to only high-media names,
- improves cross-sectional fairness and coverage,
- controls query count and execution feasibility.

### Parameters

| Parameter | Description |
|-----------|-------------|
| `volume_query_mode` | `"three_pass"` or `"iterative"`; how volume is queried when building baskets. |
| `volume_correction` | Optional `(percentage, threshold)` to reduce estimated volumes for companies above threshold; helps balance basket sizes. |
| `source_ids` | Optional list of source IDs to restrict search (e.g. specific publishers). |
| `min_period_days` | Minimum days per time window; constrains time splitting so baskets stay within API limits. |
| `max_iterations_per_batch` | Max iterations per batch in `"iterative"` volume mode (default 10). |
| `reranker_enabled` | When true, a cross-encoder reranker reorders/filters chunks by relevance; when false, initial retrieval scores are used. |
| `reranker_threshold` | Relevance cutoff (0–1) when reranker is enabled; higher = fewer, more relevant chunks. |

**Reranker recommendations:** For **large-scale datasets** (broad universes, many themes, or high total chunk counts), we **recommend setting `reranker_enabled=False`**. This increases **recall and coverage** by avoiding extra filtering and reordering that can drop chunks and add latency; you keep the full initial retrieval set for downstream aggregation and signal construction.

Output artifact:
- basketized search plans with expected chunk counts and query metadata.



In [ ]:
# Create search plans for each sub-theme
plans_list = []
for i, summary in enumerate(terminal_labels_and_summaries.values()):
    plan = plan_search(
        text=summary,
        universe=universe_csv,
        start_date=start_date,
        end_date=end_date,
        volume_query_mode="iterative",
        max_iterations_per_batch=10,
        min_period_days=90,  # Minimum time window size for each basket
        min_entities_per_basket=10,
    )

    
    # Save plan to file for reproducibility and debugging
    plan_file = f"basic_search_plan_node_{i}.json"
    plans_list.append(plan_file)
    save_plan(plan, plan_file)

    print(f"Theme {i}: {list(terminal_labels_and_summaries.keys())[i]}")
    print(f"  Total expected chunks: {plan['chunk_upper_bound_estimate']:,}")
    print(f"  Number of baskets: {len(plan['baskets'])}")



2026-04-20 09:41:20,859 - INFO - Planning search for text: 'Companies will face production delays due to semiconductor supply shortages disrupting global production.'
2026-04-20 09:41:20,860 - INFO - Date range: 2021-01-01 to 2021-06-30
2026-04-20 09:41:20,862 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
2026-04-20 09:41:20,862 - INFO - Loaded 4731 companies from universe
2026-04-20 09:41:20,869 - INFO - Using SmartBatchingPlanner with adaptive time-window splitting
2026-04-20 09:41:20,871 - INFO - Loaded 4731 entity IDs from id_name_mapping_us_top_3000.csv
PHASE 1: Querying full period for all companies (2021-01-01 to 2021-06-30)
         Mode: iterative
    [ITERATIVE MODE] Querying 4731 companies in 10 batches of 500 across 1 date sub-range(s) (10 work items, max_workers=8)
    Each batch iterates until no new companies are found (max 10 iterations)
      Batch 6/10, Iter 1: Found 299 new companies, 201 remaining
      Batch 7/10, Iter 1: Found 286 new compan

## 3. Search Execution

Execute planned baskets and collect raw documents/chunks.

Why this matters:
- converts planning assumptions into realized retrieval coverage,
- creates the base evidence set for both signal families.

Output artifact:
- `results_list` (deduplicated retrieval outputs per sub-theme).



In [ ]:
# Execute searches for all sub-themes
results_list = []
for plan_name in plans_list:
    plan = load_plan(plan_name)
    
    results_raw = execute_search(
        search_plan=plan,
        chunk_percentage=chunk_percentage,  # Sample this percentage of estimated chunks
        requests_per_minute=450,  # API rate limit
        max_workers=10,
    )

    # Deduplicate: merge chunks from same document across different baskets
    results = deduplicate_documents(results_raw)
    
    # Save raw results for debugging/reproducibility
    with open(f'basic_results_{plan_name}', 'w') as f:
        json.dump(results, f)
    results_list.append(results)

    print(f"\n Search complete for {plan_name}!")
    print(f"   Retrieved {len(results):,} deduplicated documents")



2026-04-20 09:42:51,872 - INFO - Plan loaded from search_plan_node_0.json
2026-04-20 09:42:51,872 - INFO - Executing search with 20.0% of chunks
2026-04-20 09:42:51,873 - INFO - Total maximum expected chunks: 58,665
2026-04-20 09:42:51,873 - INFO - Searching 239 baskets
2026-04-20 09:42:54,022 - INFO - Basket basket_101_medium_20210101_20210401: Retrieved 96 documents with 166 chunks
2026-04-20 09:42:54,081 - INFO - Basket basket_106_medium_20210415_20210630: Retrieved 110 documents with 150 chunks
2026-04-20 09:42:54,127 - INFO - Basket basket_105_medium_20210101_20210414: Retrieved 90 documents with 153 chunks
2026-04-20 09:42:54,204 - INFO - Basket basket_103_medium_20210101_20210423: Retrieved 109 documents with 157 chunks
2026-04-20 09:42:54,209 - INFO - Basket basket_102_medium_20210402_20210630: Retrieved 99 documents with 157 chunks
2026-04-20 09:42:54,374 - INFO - Basket basket_100_medium_20210427_20210630: Retrieved 104 documents with 165 chunks
2026-04-20 09:42:54,467 - INFO

## 3.1 Standard Full-Grid Search (Baseline)

Optional baseline path without smart batching.

Use this for:
- diagnostics,
- sanity checks,
- cost/coverage comparison against smart batching.

In large universes, this path is typically much less query-efficient.



In [134]:
# Standard Full Grid Search

#fg_search_batch_size = 500  # number of companies per API request (API maximum)
#fg_search_max_chunks_per_request = 1000  # max chunks per query (API limit)
#fg_search_requests_per_minute = 50

#results_list = []
#for i, summary in enumerate(terminal_labels_and_summaries.values()):
#    results_raw = execute_full_grid_search(
#        text=summary,
#        universe_csv_path=universe_csv,
#        start_date=start_date,
#        end_date=end_date,
#        batch_size=fg_search_batch_size,
#        session=session,
#        requests_per_minute=fg_search_requests_per_minute,
#        max_chunks_per_request=fg_search_max_chunks_per_request,
#    )

    # Deduplicate: merge chunks from same document across different batches
#    results = deduplicate_documents(results_raw)

#    results_list.append(results)
#    label = list(terminal_labels_and_summaries.keys())[i]
#    print(f"\n Normal search complete for theme {i}: {label}")
#    print(f"   Retrieved {len(results):,} deduplicated documents")




## 4. Results Aggregation and Labeling

Merge outputs from all sub-theme searches and deduplicate at chunk level.

Why this matters:
- preserves recall from multi-query retrieval,
- avoids double-counting repeated chunk evidence,
- keeps thematic provenance (`label`, `theme`) for auditability.

Output artifact:
- unified chunk-level dataset for entity mapping and signal construction.



In [9]:
# Convert results to DataFrames and tag with sub-theme labels
df_results_list = []
labels_summaries = list(terminal_labels_and_summaries.items())

for results, (label, theme_text) in zip(results_list, labels_summaries):
    df_exploded_by_chunk = convert_to_dataframe(results)
    df_exploded_by_chunk['label'] = label
    df_exploded_by_chunk['theme'] = theme_text
    df_results_list.append(df_exploded_by_chunk)

# Concatenate all results
df_concat = pd.concat(df_results_list, ignore_index=True)

# Aggregate by chunk (chunk_text): same chunk from different themes gets label/theme as lists
df_exploded_by_chunk = aggregate_results_by_chunk(df_concat)

print(f"Total unique chunks: {len(df_exploded_by_chunk):,}")

Total unique chunks: 42,065


In [136]:
# Save checkpoint (uncomment to save - useful for long-running workflows)
# with open('df_exploded_by_chunk.pkl', 'wb') as f:
#     pickle.dump(df_exploded_by_chunk, f)

# Load from checkpoint (uncomment to skip search if already completed)
# with open('df_exploded_by_chunk.pkl', 'rb') as f:
#     df_exploded_by_chunk = pickle.load(f)


## 5. Entity Resolution and Company Filtering

Map text evidence to canonical entities and restrict to the investable universe.

**This step uses only the portfolio/universe mapping (no API calls).** It keeps detections whose entity IDs are already in the universe CSV, so the **basic signal** can be built without Knowledge Graph (KG) calls. Full resolution of *unknown* entities (via the KG) is done later, only when building the **advanced signal** (Section 5.3).

Why this matters:
- makes outputs portfolio-compatible,
- removes orphan/non-investable detections,
- stabilizes cross-sectional comparability.

Output artifact:
- `df_detection` (universe-only companies), then `df_exploded_by_entity` with one row per `(chunk, entity)` for the basic signal.



In [10]:
# Filter to companies already in the universe (no KG / no API calls)
# Use empty other_companies so only universe IDs are kept; KG is used later for the advanced signal
df_exploded_by_chunk_only_companies = map_create_only_companies_column(
    df_exploded_by_chunk,
    universe_csv,
    other_companies=[],
)
df_detection = keep_only_companies_in_detections(df_exploded_by_chunk_only_companies)

### 5.1 Explode by Entity

Transform chunk-level results into `(chunk, entity)` observations.

Each row becomes an exposure candidate with:
- timestamp metadata,
- relevance/sentiment fields,
- entity identity required for cross-sectional aggregation.



In [11]:
# Explode: one row per (chunk, entity) pair, filtered by universe
df_exploded_by_entity = explode_to_dataframe(
    df_detection,
    universe_csv=universe_csv
)

print(f"Exploded DataFrame shape: {df_exploded_by_entity.shape}")
print(f"Unique entities: {df_exploded_by_entity['entity_id'].nunique()}")
df_exploded_by_entity.head()

Exploded DataFrame shape: (55239, 21)
Unique entities: 3136


,entity_name,entity_id,chunk_text,label,theme,date,doc_timestamp,doc_id,headline,source_id,...,source_rank,chunk_index,chunk_relevance,chunk_sentiment,entity_ids,detections,url,reporting_entities,entity_ids_companies,companies_detection
0,Humana Inc.,00067A,Shares of CVS Health (CVS) are up about 1% in ...,[Production Delays],[Companies will face production delays due to ...,2021-01-04,2021-01-04 17:34:12+00:00,A65A658D315467759A8107B743084D31,CVS Health up 1% after Amazon-Berkshire-JPMorg...,B5235B,...,RANK_1,1,0.000342,-0.31,"[86A1B9, 385C0C, 457FA4, 85ED51, C4F920, 14A15...","[{'id': '86A1B9', 'start': 430, 'end': 440, 't...",,[],"[86A1B9, 7AB859, 168A5D, 205AD5, 00067A, 61988...","[{'id': '86A1B9', 'start': 430, 'end': 440, 't..."
1,Humana Inc.,00067A,"8\nchains, could materially and adversely disr...",[Production Delays],[Companies will face production delays due to ...,2021-02-03,2021-02-03 14:00:00+00:00,A4DFAB4CEB11FAF2490860DBCCADDF5A,"Humana Inc: Q4 2020 Earnings Call on Feb 3, 20...",28DED6,...,RANK_1,40,0.007329,-0.88,"[0A8D2C, 58DAFA, 58DAFA, E735C9, E735C9, 929A6...","[{'id': '0A8D2C', 'start': 449, 'end': 457, 't...",https://files.quartr.com/reports/ab414-2024-04...,[],"[00067A, 00067A, 00067A, 00067A, 00067A, 00067...","[{'id': '00067A', 'start': 91, 'end': 97, 'typ..."
2,Humana Inc.,00067A,The outbreak of COVID-19 has severely impacted...,[Production Delays],[Companies will face production delays due to ...,2021-02-03,2021-02-03 11:30:01+00:00,525B7EBB32D71F086A2E1BF1DD509DD5,Humana Reports Fourth Quarter 2020 Financial R...,5A5702,...,RANK_1,49,0.045126,-0.87,"[58DAFA, 5F36FA, 929A63, E735C9, 81C098, 00067...","[{'id': '58DAFA', 'start': 16, 'end': 24, 'typ...",,[],"[00067A, 00067A, 00067A]","[{'id': '00067A', 'start': 109, 'end': 115, 't..."
3,Humana Inc.,00067A,"The spread and impact of COVID-19, or actions ...",[Production Delays],[Companies will face production delays due to ...,2021-02-03,2021-02-03 14:00:00+00:00,A4DFAB4CEB11FAF2490860DBCCADDF5A,"Humana Inc: Q4 2020 Earnings Call on Feb 3, 20...",28DED6,...,RANK_1,39,0.020160,-0.78,"[6F2822, B0F242, 00067A, C590B1, ED3F19, E1B51...","[{'id': '6F2822', 'start': 235, 'end': 250, 't...",https://files.quartr.com/reports/ab414-2024-04...,[],[00067A],"[{'id': '00067A', 'start': 120, 'end': 126, 't..."
4,Humana Inc.,00067A,"The spread and impact of COVID-19, or actions ...",[Production Delays],[Companies will face production delays due to ...,2021-02-03,2021-02-03 11:30:01+00:00,525B7EBB32D71F086A2E1BF1DD509DD5,Humana Reports Fourth Quarter 2020 Financial R...,5A5702,...,RANK_1,47,0.027010,-0.86,"[FF98F9, ED3F19, 5F3B00, 6F2822, 00067A, 00067...","[{'id': 'FF98F9', 'start': 482, 'end': 501, 't...",,[],"[00067A, 00067A]","[{'id': '00067A', 'start': 120, 'end': 126, 't..."


### 5.2 Basic Signal Construction (High Recall, Fast)

This section builds the **basic signal family** directly from retrieval outputs.

Definition per entity-day:
- `score = Σ(relevance × sentiment)` over deduplicated `(entity, doc, chunk)` rows
- `volume = count(entity, doc, chunk)`

Interpretation:
- high recall, low-latency thematic proxy,
- lower precision than LLM-verified path,
- useful for broad hypothesis screening and rapid iteration.



In [12]:
# Build daily basic entity-level signals from deduplicated chunk rows

relevance_col = "chunk_relevance" if "chunk_relevance" in df_exploded_by_entity.columns else "relevance"
sentiment_col = "chunk_sentiment" if "chunk_sentiment" in df_exploded_by_entity.columns else "sentiment"
chunk_col = "chunk_index" if "chunk_index" in df_exploded_by_entity.columns else "chunk_cnum"
date_col = "doc_timestamp" if "doc_timestamp" in df_exploded_by_entity.columns else "date"

required_cols = {"entity_id", "doc_id", chunk_col, relevance_col, sentiment_col, date_col}
missing_cols = [c for c in required_cols if c not in df_exploded_by_entity.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for daily basic signals: {missing_cols}")

# 1) Keep rows with valid relevance/sentiment and parse date
basic_long = df_exploded_by_entity[
    ["entity_id", "doc_id", chunk_col, date_col, relevance_col, sentiment_col]
].copy()
basic_long = basic_long.dropna(subset=[relevance_col, sentiment_col, date_col])

# Assign trading date so we aggregate before 15:30 NY (no look-ahead for close-to-close NYSE).
# Assume source timestamps are UTC; convert to Eastern then offset so 15:30 NY = day boundary.
eastern = ZoneInfo("America/New_York")
dt_utc = pd.to_datetime(basic_long[date_col], errors="coerce", utc=True)
dt_ny = dt_utc.dt.tz_convert(eastern)
# Trading date T = [15:30 NY on T-1, 15:30 NY on T). Add 8.5h so 15:30 becomes midnight.
basic_long["date"] = (dt_ny + pd.Timedelta(hours=8.5)).dt.date
basic_long = basic_long.dropna(subset=["date"])

# 2) Deduplicate per entity-date-doc-chunk
basic_long = basic_long.drop_duplicates(subset=["entity_id", "date", "doc_id", chunk_col])

# 3) Aggregate by entity and date
if basic_long.empty:
    df_basic_signals = pd.DataFrame(columns=["rp_entity_id", "date", "score", "volume", "entity_name"])
else:
    basic_long["score_component"] = basic_long[relevance_col] * basic_long[sentiment_col]

    entity_daily_agg = (
        basic_long.groupby(["entity_id", "date"], as_index=False)
        .agg(
            score=("score_component", "sum"),
            volume=("entity_id", "size"),
        )
        .rename(columns={"entity_id": "rp_entity_id"})
    )

    # Enrich with entity names from the selected universe CSV
    df_universe = pd.read_csv(universe_csv)
    df_universe = df_universe.rename(columns={"id": "rp_entity_id", "name": "entity_name"})

    df_basic_signals = entity_daily_agg.merge(
        df_universe[["rp_entity_id", "entity_name"]],
        on="rp_entity_id",
        how="left",
    )

    df_basic_signals = df_basic_signals[["rp_entity_id", "date", "score", "volume", "entity_name"]]
    df_basic_signals = df_basic_signals.sort_values(["rp_entity_id", "date"]).reset_index(drop=True)

# Save
basic_signals_path = "df_basic_signals_daily.csv"
df_basic_signals.to_csv(basic_signals_path, index=False)

print(f"Deduplicated entity-date-doc-chunk rows: {len(basic_long)}")
print(f"Entity-date rows with basic signals: {len(df_basic_signals)}")
print(f"Saved: {basic_signals_path}")
df_basic_signals.head()


Deduplicated entity-date-doc-chunk rows: 55213
Entity-date rows with basic signals: 24728
Saved: df_basic_signals_daily.csv


,rp_entity_id,date,score,volume,entity_name
0,00067A,2021-01-04,-0.000106,1,Humana Inc.
1,00067A,2021-02-03,-0.084662,4,Humana Inc.
2,00067A,2021-04-28,-0.098410,5,Humana Inc.
3,001F1B,2021-01-08,-0.006948,2,PriceSmart Inc.
4,001F1B,2021-04-09,-0.011951,4,PriceSmart Inc.


## Summary

1. **Basic (relevance × sentiment):**
   - faster, broader, higher recall.

For robust research practice:
- validate timing assumptions (`doc_timestamp` -> `date_nyse_1530`),
- test signal behavior across themes and market regimes,
- treat in-sample performance as a **sanity check**, not proof of persistent alpha.

Recommended next steps:
- compare basic signal information ratio and turnover,
- run robustness tests across alternate themes and windows,
- evaluate implementation frictions before any live deployment.

